<center><h1> NON PR Streak </h1></center>

The following code is a slightly modified version of one written by [Thomas Stokes](https://github.com/ThomasStokes1998), WCAID [2017STOK03](https://www.worldcubeassociation.org/persons/2017STOK03). Full credit goes to him.

It calculates the longest streak of consecutive WCA competitions without a <b>personal record</b> (PR) for each competitor.

<small>Startcomp is the first competition of the streak.<br>Endcomp is the competition that breaks the streak (N+1).</small>

### Imports & functions

In [1]:
import pandas as pd
from datetime import date
import numpy as np

low_memory = False
results = pd.read_csv('WCA_export_Results.tsv', delimiter='\t')
competitions = pd.read_csv('WCA_export_Competitions.tsv', delimiter='\t')


In [2]:
def mbfPoints(p: int, oldstyle: bool=False) -> float:
    if oldstyle:
        tt = p % 100_000
        ss = 199 - p // 10_000_000
        aa = (p // 100_000) % 100
        if tt == 99_999:
            tt = min(600 * aa, 3_600)
        return ss - aa + 1 - tt / 3_600
    else:
        dd = 99 - p // 10_000_000
        mm = p % 100
        tt = (p // 100) % 100_000
        aa = dd + 2 * mm
        if tt == 99_999:
            tt = min(600 * aa, 3_600)
        if aa < 6:
            return dd + 1 - tt / (600 * aa)
        else:
            return dd + 1 - tt / 3_600


### Dictionaries to keep track of all the data

In [3]:
compdates = {"ongoing":date.today()}
for i, comp in enumerate(competitions.id):
    compdates[comp] = date(competitions.year[i], competitions.month[i], competitions.day[i])


In [4]:
personnames = {}
maxstreaks = {}
maxstart = {}
maxend = {}
currentcomp = {}
startcomps = {}
currentstreak = {}
gotpr = {}
personprs = {}

### Code

In [5]:
def updatedicts(event: str, wcaid: str, single: int, average: int):
    # Add a new event 
    if event not in personprs[wcaid]:
        if event == "333mbf":
            personprs[wcaid][event] = 0
        elif event == "333mbo":
            personprs[wcaid][event] = 0
        else:
            personprs[wcaid][event] = [360_000, 360_000]
    # Check if a new PR has been set
    if event == "333mbf":
        if single > 0 and personprs[wcaid][event] <= mbfPoints(single):
            personprs[wcaid][event] = mbfPoints(single)
            gotpr[wcaid] = True
    elif event == "333mbo":
        if single > 0 and personprs[wcaid][event] <= mbfPoints(single, True):
            personprs[wcaid][event] = mbfPoints(single, True)
            gotpr[wcaid] = True
    else:
        if single > 0 and personprs[wcaid][event][0] >= single:
            personprs[wcaid][event][0] = single
            gotpr[wcaid] = True
        if average > 0 and personprs[wcaid][event][1] >= average:
            personprs[wcaid][event][1] = average
            gotpr[wcaid] = True

Change the country in the first row below to the country of choice or to "world" for global PR straks

In [6]:
def main(country: str = "World"):
    if country == "World":
        res = results
    else:
        res = results[results.personCountryId == country].reset_index(drop="index")
    # Order the res dataframe by date
    res["date"] = res["competitionId"].apply(lambda x: compdates[x])
    res = res.sort_values("date").reset_index(drop="index")
    for i, wcaid in enumerate(res.personId):
        comp = res.competitionId[i]
        event = res.eventId[i]
        single = res.best[i]
        average = res.average[i]
        if wcaid not in personnames:
            # Initial values for a new id
            personnames[wcaid] = res.personName[i]
            maxstreaks[wcaid] = 0
            maxstart[wcaid] = "none"
            maxend[wcaid] = "none"
            startcomps[wcaid] = "none"
            currentcomp[wcaid] = comp
            currentstreak[wcaid] = 0
            gotpr[wcaid] = False
            personprs[wcaid] = {}
        if currentcomp[wcaid] == comp:
            updatedicts(event, wcaid, single, average)
        else:
            if gotpr[wcaid]:
                c = currentstreak[wcaid]
                if c > maxstreaks[wcaid]:
                    maxstreaks[wcaid] = c
                    maxstart[wcaid] = startcomps[wcaid]
                    maxend[wcaid] = currentcomp[wcaid]
                currentstreak[wcaid] = 0
            else:
                if currentstreak[wcaid] == 0:
                    startcomps[wcaid] = currentcomp[wcaid]
                currentstreak[wcaid] += 1
            # Reset values for a new competition
            gotpr[wcaid] = False
            currentcomp[wcaid] = comp
            updatedicts(event, wcaid, single, average)
    # Update ongoing streaks
    for wcaid in gotpr:
        if maxstart[wcaid] == "none":
            maxstart[wcaid] = currentcomp[wcaid]
            maxend[wcaid] = currentcomp[wcaid]
        if gotpr[wcaid]:
            c = currentstreak[wcaid]
            if c > maxstreaks[wcaid]:
                maxstreaks[wcaid] = c
                maxstart[wcaid] = startcomps[wcaid]
                maxend[wcaid] = currentcomp[wcaid]
        else:
            if currentstreak[wcaid] == 0:
                startcomps[wcaid] = currentcomp[wcaid]
            currentstreak[wcaid] += 1
            c = currentstreak[wcaid]
            if c > maxstreaks[wcaid]:
                maxstreaks[wcaid] = c
                maxstart[wcaid] = startcomps[wcaid]
                maxend[wcaid] = "ongoing"
            
                
    # Create the dataframe
    names = [personnames[wcaid] for wcaid in gotpr]
    maxstreak = [maxstreaks[wcaid] for wcaid in gotpr]
    maxstarts = [maxstart[wcaid] for wcaid in gotpr]
    maxends = [maxend[wcaid] for wcaid in gotpr]
    maxdays = [(compdates[maxends[i]] - compdates[maxstarts[i]]).days for i, _ in enumerate(maxends)]
    df = pd.DataFrame({"name":names, "GTS":maxstreak, "startcomp":maxstarts, "endcomp":maxends, 
    "days":maxdays})
    df = df.sort_values("GTS", ascending=False).reset_index(drop="index")
    df.to_csv(f"maxnonprstreak{country}.csv", index=False)


### Result

In [7]:
if __name__ == "__main__":
    # Longest PR Streak in Country
    main()

Pr streaks

In [8]:
a = pd.read_csv('maxnonprstreakWorld.csv') #change the name if you change the country to 'maxprstreak[country].csv'
a.index += 1
a.head(20)

,name,GTS,startcomp,endcomp,days
1,Natán Riggenbach,39,SpeedsolvingPucallpa2016,JaenMountainCubes2018,980
2,Matthew Dickman,37,CubingUSANationals2019,SpeedcubeBajaOpen2022,1171
3,Dave Campbell,33,HillsdaleWinter2012,LondonLimitedSpring2017,1848
4,Shelley Chang,28,USNationals2012,CubingSkillcon2015,1191
5,Karolina Wiącek,28,PolishNationals2014,GdanskRubiksCubeDay2016,889
6,Finn Ickler,27,TirolOpen2019,ongoing,1053
7,Yulia Sidorova,26,SuisseToyFastFingers2017,ongoing,1844
8,Donglei Li (李冬雷),24,XianCherryBlossom2017,ongoing,2039
9,Ryan DeLine,23,PleaseBeAsQuietAsACat2018,ongoing,1683
10,Tim McMahon,21,MelbourneCubeDay2015,ongoing,2551


Ongoing PR streaks

In [9]:
a[a['endcomp']=='ongoing'].reset_index(drop=True).head(20)

,name,GTS,startcomp,endcomp,days
0,Finn Ickler,27,TirolOpen2019,ongoing,1053
1,Yulia Sidorova,26,SuisseToyFastFingers2017,ongoing,1844
2,Donglei Li (李冬雷),24,XianCherryBlossom2017,ongoing,2039
3,Ryan DeLine,23,PleaseBeAsQuietAsACat2018,ongoing,1683
4,Tim McMahon,21,MelbourneCubeDay2015,ongoing,2551
5,Tyson Mao (毛台勝),20,WC2009,ongoing,4771
6,Javier Cabezuelo Sánchez,19,SpanishChampionship2016,ongoing,2216
7,Alberto Masó Molina,19,ElEscorialOpen2018,ongoing,1599
8,Camilla Jul Nielsson,19,DanishOpen2014,ongoing,3132
9,Rama Temmink,18,DutchOpen2012,ongoing,3650
